In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

In [2]:
df = pd.read_csv("data/filtered_sensor_data.csv", header=0)

In [3]:
data = df[["Timestamp_ms","Angle_raw","Angle_low","Angle_ab","Vel_raw","Vel_out"]].copy()
for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
data = data.dropna().sort_values("Timestamp_ms").reset_index(drop=True)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Angle (rad)", "Velocity (rad/s)"),
)

fig.add_trace(
    go.Scatter(x=data["Timestamp_ms"]/1000, y=data["Angle_raw"], name="Angle Raw"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=data["Timestamp_ms"]/1000, y=data["Angle_low"], name="Angle Low"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=data["Timestamp_ms"]/1000, y=data["Angle_ab"], name="Angle Ab"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=data["Timestamp_ms"]/1000, y=data["Vel_raw"], name="Velocity Raw"),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=data["Timestamp_ms"]/1000, y=data["Vel_out"], name="Velocity Out"),
    row=2, col=1
)

fig.update_xaxes(title_text="Timestamp (s)", row=2, col=1)
fig.update_yaxes(title_text="rad", row=1, col=1)
fig.update_yaxes(title_text="rad/s", row=2, col=1)
fig.update_layout(height=600, showlegend=False)
fig.update_layout(
    title_text="Unfiltered Sensor Readings",
    showlegend=True
)
fig.show()

## Do a FFT

In [4]:
t = data["Timestamp_ms"].to_numpy() / 1000  # seconds
pend_ang = data["Angle_raw"].to_numpy()

dt = np.median(np.diff(t))
t_uniform = np.arange(t[0], t[-1], dt)
pend_ang_uniform = np.interp(t_uniform, t, pend_ang)

# Remove the mean and apply a Hann window
signal = pend_ang_uniform - np.mean(pend_ang_uniform)
window = np.hanning(len(signal))
spectrum = np.fft.rfft(signal * window)
frequencies = np.fft.rfftfreq(len(signal), d=dt)

# Normalize the one-sided amplitude spectrum
amplitude = 2 * np.abs(spectrum) / np.sum(window)
amplitude[0] /= 2
if len(signal) % 2 == 0:
    amplitude[-1] /= 2

fft_fig = go.Figure(
    go.Scatter(x=frequencies, y=amplitude, mode="lines", name="Angle FFT")
)
fft_fig.update_layout(
    title="FFT of Angle",
    xaxis_title="Frequency (Hz)",
    yaxis_title="Amplitude (rad)",
)
fft_fig.show()

## Time Step Analysis

In [5]:
timestamp_diffs = data["Timestamp_ms"].diff().dropna()

fig = go.Figure(go.Histogram(x=timestamp_diffs, nbinsx=50))
fig.update_layout(
    title="Distribution of Timestamp Differences",
    xaxis_title="Timestamp difference (ms)",
    yaxis_title="Count",
)
fig.show()

In [6]:
timestamp_diffs = data["Timestamp_ms"].diff()

diff_fig = go.Figure(
    go.Scatter(
        x=data.loc[timestamp_diffs.notna(), "Timestamp_ms"] / 1000,
        y=timestamp_diffs.dropna(),
        mode="lines+markers",
        name="Timestamp difference",
    )
)
diff_fig.update_layout(
    title="Timestamp Difference Over Time",
    xaxis_title="Time (s)",
    yaxis_title="Timestamp difference (ms)",
)
diff_fig.show()